# Recipe-Planner Agent



A real, runnable "recipe planner" agent -- the notebook version of the example that accompanies the course's [Recipe-Planner Agent](https://github.com/abderrahim-lectures/python-data-analysis-course/blob/main/docs/projects/recipe-planner-agent/index.md) project, built with LangChain's [`deepagents`](https://github.com/langchain-ai/deepagents).



Given a list of ingredients you have on hand, this agent suggests 2-3 real meals you could make -- grounded in a local recipe database (embedded below), never invented by the model -- and produces a shopping list for the best suggestion.



This mirrors [`examples/recipe-planner-agent/planner.py`](https://github.com/abderrahim-lectures/python-data-analysis-course/blob/main/examples/recipe-planner-agent/planner.py) and [`recipes.py`](https://github.com/abderrahim-lectures/python-data-analysis-course/blob/main/examples/recipe-planner-agent/recipes.py) from the course repo, adapted to run cell-by-cell here.



**Note:** model names and library APIs in this space change fast -- if a cell below errors, check the [lesson's Setup section](https://github.com/abderrahim-lectures/python-data-analysis-course/blob/main/docs/projects/recipe-planner-agent/index.md#setup) for what may have drifted.

In [ ]:
!pip install -q deepagents langchain-openai python-dotenv

## The recipe database



A small, local list of recipes -- the structured data the agent is grounded in, instead of letting it hallucinate recipes from its training data. Each recipe is a plain dict: a name, its ingredient list (lowercase, no quantities), and short instructions.

In [ ]:
RECIPES = [
    {
        "name": "Tomato Egg Stir-Fry",
        "ingredients": ["eggs", "tomatoes", "garlic", "salt", "oil"],
        "instructions": "Scramble the eggs, set aside. Saute garlic and chopped tomatoes "
        "until soft, stir the eggs back in, season with salt.",
    },
    {
        "name": "Garlic Butter Pasta",
        "ingredients": ["pasta", "butter", "garlic", "parmesan", "salt"],
        "instructions": "Boil the pasta. Melt butter with minced garlic, toss the pasta in "
        "it, top with grated parmesan and salt.",
    },
    {
        "name": "Classic Grilled Cheese",
        "ingredients": ["bread", "cheese", "butter"],
        "instructions": "Butter one side of each bread slice, add cheese between the "
        "unbuttered sides, grill in a pan until golden on both sides.",
    },
    {
        "name": "Simple Fried Rice",
        "ingredients": ["rice", "eggs", "soy sauce", "onion", "oil"],
        "instructions": "Scramble the eggs and set aside. Fry chopped onion in oil, add "
        "cooked rice, stir in soy sauce and the eggs.",
    },
    {
        "name": "Chickpea Salad",
        "ingredients": ["chickpeas", "cucumber", "tomatoes", "olive oil", "lemon", "salt"],
        "instructions": "Drain the chickpeas, dice the cucumber and tomatoes, toss "
        "everything with olive oil, lemon juice, and salt.",
    },
    {
        "name": "Peanut Butter Banana Toast",
        "ingredients": ["bread", "peanut butter", "banana"],
        "instructions": "Toast the bread, spread peanut butter, top with sliced banana.",
    },
    {
        "name": "Basic Vegetable Soup",
        "ingredients": ["carrots", "onion", "celery", "vegetable broth", "salt"],
        "instructions": "Saute chopped carrots, onion, and celery, add vegetable broth, "
        "simmer 15 minutes, season with salt.",
    },
    {
        "name": "Tuna Sandwich",
        "ingredients": ["bread", "tuna", "mayonnaise", "onion"],
        "instructions": "Mix drained tuna with mayonnaise and finely chopped onion, spread "
        "between bread slices.",
    },
    {
        "name": "Honey Mustard Chicken",
        "ingredients": ["chicken breast", "honey", "mustard", "salt", "oil"],
        "instructions": "Mix honey and mustard, coat the chicken breast, season with salt, "
        "pan-fry in oil until cooked through.",
    },
    {
        "name": "Guacamole",
        "ingredients": ["avocado", "lime", "onion", "tomatoes", "salt"],
        "instructions": "Mash the avocado, stir in lime juice, finely chopped onion and "
        "tomatoes, season with salt.",
    },
    {
        "name": "Oatmeal with Banana",
        "ingredients": ["oats", "milk", "banana", "honey"],
        "instructions": "Simmer oats in milk until thick, top with sliced banana and a "
        "drizzle of honey.",
    },
    {
        "name": "Caprese Salad",
        "ingredients": ["tomatoes", "mozzarella", "basil", "olive oil", "salt"],
        "instructions": "Slice tomatoes and mozzarella, layer with basil leaves, drizzle "
        "with olive oil, season with salt.",
    },
    {
        "name": "Black Bean Tacos",
        "ingredients": ["tortillas", "black beans", "onion", "lime", "salt"],
        "instructions": "Warm the black beans with chopped onion and salt, spoon into "
        "tortillas, finish with a squeeze of lime.",
    },
]

## API key



This defaults to [GitHub Models](https://docs.github.com/en/github-models) (free tier, uses a GitHub account you already have -- see the six-provider table in the lesson's Setup section for other options). Paste a [GitHub personal access token](https://github.com/settings/tokens) below; it is never displayed or stored on disk, only kept in memory for this session.

In [ ]:
import os

from getpass import getpass



os.environ["GITHUB_TOKEN"] = getpass("Enter your GitHub token (or other provider's API key): ")

## The search tool



This is the agent's one source of truth for what recipes exist. It searches `RECIPES` for the entries that best overlap with the ingredients on hand, and reports exactly what's missing from each -- so the agent never has to invent a recipe or guess at a shopping list.

In [ ]:
def search_recipes_by_ingredients(ingredients: list[str]) -> str:

    """Search the local recipe database for recipes that best match the given ingredients.



    `ingredients` should be a list of ingredient names the caller already

    has on hand (e.g. ["eggs", "tomatoes", "garlic"]). Returns the top

    matching recipes, ranked by how many of their ingredients are already

    covered, each with its full ingredient list and the ingredients still

    missing -- so a shopping list can be built from the result without

    guessing. Returns a plain "no matches" message if nothing overlaps at

    all, so the caller never has to invent a recipe out of thin air.

    """

    have = {i.strip().lower() for i in ingredients}

    scored = []

    for recipe in RECIPES:

        needed = {i.lower() for i in recipe["ingredients"]}

        overlap = have & needed

        if not overlap:

            continue

        missing = sorted(needed - have)

        scored.append((len(overlap), recipe, missing))



    if not scored:

        return "No matching recipes found in the database for those ingredients."



    scored.sort(key=lambda row: row[0], reverse=True)

    top = scored[:5]



    lines = []

    for _, recipe, missing in top:

        missing_text = ", ".join(missing) if missing else "nothing -- you have it all!"

        lines.append(

            f"- {recipe['name']} | full ingredient list: {', '.join(recipe['ingredients'])} "

            f"| missing: {missing_text}"

        )

    return "Matching recipes (best match first):\n" + "\n".join(lines)

## Build the agent



The system prompt explicitly forbids inventing a recipe the tool didn't return -- every suggestion has to trace back to a real entry in `RECIPES`.

In [ ]:
from deepagents import create_deep_agent

from langchain_openai import ChatOpenAI



SYSTEM_PROMPT = """You are a helpful recipe-planning assistant.



You have exactly one source of truth for what recipes exist: the

search_recipes_by_ingredients tool. Never invent, guess, or recall a recipe

from your own training data -- only suggest recipes that tool actually

returned in its results for this conversation.



When a student lists what they have on hand:

1. Call search_recipes_by_ingredients with that ingredient list.

2. Suggest 2-3 recipes from the tool's results, explaining briefly why each

   is a good fit (how much they already have).

3. If the tool returns no matches, say so plainly and suggest the student

   try listing a few more ingredients -- do not make up a recipe to fill

   the gap.

4. If asked to build a shopping list for a specific recipe, use the

   "missing" ingredients the tool already reported for that recipe -- don't

   recompute or guess at what's missing.

"""



model = ChatOpenAI(

    model="gpt-4o-mini",  # confirm this still has a free tier before running

    api_key=os.environ["GITHUB_TOKEN"],

    base_url="https://models.github.ai/inference",

)



agent = create_deep_agent(

    model=model,

    tools=[search_recipes_by_ingredients],

    system_prompt=SYSTEM_PROMPT,

)



def ask(agent, messages: list[dict]) -> dict:

    """Run the conversation so far through the agent and return the raw result."""

    return agent.invoke({"messages": messages})

## Ask for meal suggestions



Given a real sample ingredient list, ask the agent what it can make, then ask for a shopping list for its top suggestion.

In [ ]:
conversation: list[dict] = []



on_hand = "I have eggs, tomatoes, garlic, bread, and cheese. What can I make?"

print("You:", on_hand)

conversation.append({"role": "user", "content": on_hand})

result = ask(agent, conversation)

conversation = result["messages"]  # carry the full history forward

print("Agent:", conversation[-1].content)



print()

follow_up = "Great, let's go with the first one -- what's my shopping list?"

print("You:", follow_up)

conversation.append({"role": "user", "content": follow_up})

result = ask(agent, conversation)

conversation = result["messages"]

print("Agent:", conversation[-1].content)